# Data Challenge 11 — Evaluating MLR & Fixing Multicollinearity (HVFHV Trips)


**Format:** Instructor Guidance → You Do (Students) → We Share (Reflection)

**Goal:** Build an MLR, evaluate it with a **train–test split**, diagnose multicollinearity with **corr** and **VIF** on the **training set**, fix issues (drop/choose features), and report **test MAE/RMSE** + **coefficient interpretations**.

**Data:** July 1, 2023 - July 15, 2023 For Hire Vehicle Data in NYC

[July For Hire Vehicles Data](https://data.cityofnewyork.us/Transportation/2023-High-Volume-FHV-Trip-Data/u253-aew4/about_data)


## Instructor Guidance

**Hint: Use the Lecture Deck, Canvas Reading, and Docs to help you with the code**

Use this guide live; students implement below.

**Docs (quick links):**
- Train/Test Split — scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
- OLS — statsmodels: https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html
- OLS Results (rsquared_adj, pvalues, resid, etc.): https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.html
- VIF — statsmodels: https://www.statsmodels.org/stable/generated/statsmodels.stats.outliers_influence.variance_inflation_factor.html
- Corr — pandas: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html

### Pseudocode Plan (Evaluation + Multicollinearity)
1) **Load CSV** → preview shape/columns; (optional) filter to **July**.
2) **Pick Y** (`base_passenger_fare`) and **candidate X’s** (e.g., `trip_miles`, `trip_time_minutes`, `tolls`, `tips` if present).
3) **Light prep** → derive `trip_time_minutes` from `trip_time` (seconds) if present; coerce only used cols to numeric; drop NA rows.
4) **Split** → `X_train, X_test, y_train, y_test` (80/20, fixed `random_state`).
5) **Diagnose on TRAIN**:
   - **Correlation matrix** (|r| > 0.7 = red flag).
   - **VIF** for each predictor (1–5 ok; >5–10+ = concerning).
6) **Fix** → drop/choose among highly correlated predictors (business logic).
7) **Fit on TRAIN only** → OLS with intercept.
8) **Predict on TEST** → compute **MAE/RMSE** (units of Y).
9) **Interpret** → unit-based coefficient sentences **holding others constant**; note any changes after fixing collinearity.
10) **Report** → table of (features kept, Adj R², MAE, RMSE) + 1-line stakeholder takeaway.


## You Do — Student Section
Work in pairs. Comment your choices briefly. Keep code simple—only coerce the columns you use.

### Step 0 — Setup & Imports

In [279]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import scipy.stats as stats
from pathlib import Path


### Step 1 — Load CSV & Preview
- Point to your For Hire Vehicle Data 
- Print **shape** and **columns**.

**Hint: You may have to drop missing values and do a force coercion to make sure the variables stay numeric (other coding assignments may help)**

In [283]:
df = pd.read_csv('/Users/beans/Desktop/FHV_072023 copy.csv')

/var/folders/3y/ldxff8k17wjcyzkq19xw1srm0000gn/T/ipykernel_6110/671824708.py:1: DtypeWarning: Columns (11,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/beans/Desktop/FHV_072023 copy.csv')


In [284]:
display(df.info())
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8324591 entries, 0 to 8324590
Data columns (total 24 columns):
 #   Column                Dtype  
---  ------                -----  
 0   hvfhs_license_num     object 
 1   dispatching_base_num  object 
 2   originating_base_num  object 
 3   request_datetime      object 
 4   on_scene_datetime     object 
 5   pickup_datetime       object 
 6   dropoff_datetime      object 
 7   PULocationID          int64  
 8   DOLocationID          int64  
 9   trip_miles            float64
 10  trip_time             object 
 11  base_passenger_fare   object 
 12  tolls                 float64
 13  bcf                   float64
 14  sales_tax             float64
 15  congestion_surcharge  float64
 16  airport_fee           float64
 17  tips                  float64
 18  driver_pay            object 
 19  shared_request_flag   object 
 20  shared_match_flag     object 
 21  access_a_ride_flag    object 
 22  wav_request_flag      object 
 23  wav_mat

None

,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0005,B03406,NaN,07/01/2023 05:34:30 PM,NaN,07/01/2023 05:37:48 PM,07/01/2023 05:44:45 PM,158,68,1.266,...,1.35,2.75,0.0,2.00,5.57,N,N,N,N,False
1,HV0003,B03404,B03404,07/01/2023 05:34:30 PM,07/01/2023 05:36:53 PM,07/01/2023 05:37:15 PM,07/01/2023 05:55:15 PM,162,234,2.350,...,1.52,2.75,0.0,3.28,13.38,N,N,NaN,N,False
2,HV0003,B03404,B03404,07/01/2023 05:34:30 PM,07/01/2023 05:35:17 PM,07/01/2023 05:35:52 PM,07/01/2023 05:44:27 PM,161,163,0.810,...,0.49,2.75,0.0,0.00,5.95,N,N,NaN,N,False
3,HV0003,B03404,B03404,07/01/2023 05:34:30 PM,07/01/2023 05:37:39 PM,07/01/2023 05:39:35 PM,07/01/2023 06:23:02 PM,122,229,15.470,...,5.17,2.75,0.0,0.00,54.46,N,N,NaN,N,True
4,HV0003,B03404,B03404,07/01/2023 05:34:30 PM,07/01/2023 05:36:06 PM,07/01/2023 05:36:39 PM,07/01/2023 05:45:06 PM,67,14,1.520,...,0.85,0.00,0.0,3.00,7.01,N,N,NaN,N,False


In [285]:
# function allows for filtering of DataFrame by column, percentile, and the threshold of column
def filterdf(df, column, percentile,filtnum):
    df = df[(df[column] <= np.percentile(df[column],percentile)) & (df[column] > filtnum)]
    return df

In [ ]:
# Coerce fare, distance to numeric safely
num_cols = ['base_passenger_fare','trip_miles','trip_time','tips']
for c in num_cols:
    df[c] = pd.to_numeric(
        df[c].astype(str).str.strip().str.replace(r'[^0-9.+\-eE]', '', regex=True),
        errors='coerce'
)
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=num_cols)

# making sure miles, trip time, and base passenger fare do not equal to 0 & removes outliers
df = filterdf(df, 'trip_miles',99.9999,0)
df = filterdf(df, 'trip_time', 99.9999, 0)
df = filterdf(df,'base_passenger_fare', 99.9999,0)

len(df)

# Lost a total of 1557 after filtering values to be above 0 and removing larger values.

8323034

### Step 2 —  Choose Target **Y** and Candidate Predictors

- Suggested **Y**: `base_passenger_fare` (USD).
- Start with **distance** and **time**; optionally add **flags** if present.
- Derive `trip_time_minutes` from `trip_time` (seconds) if available.

In [287]:
# deriving trip_time_minutes from trip_time
df['trip_time_minutes'] = df['trip_time'] / 60

# creating minutes per mile feature to use
df['min_per_mile'] = df['trip_time_minutes'] / df['trip_miles']

y = df['base_passenger_fare']
# first x for value and second x for distance
x = df[['tips','min_per_mile']]


### Step 3 — Train–Test Split

- Use a fixed `random_state` for reproducibility.
- **All diagnostics below must be done on TRAIN only.**

In [288]:
# creating a random seed and splitting our data into train and test for model testing
np.random.seed(60)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=60)


In [289]:
display(X_train)
X_train.info()


,tips,min_per_mile
3818891,0.00,2.889741
1399190,0.00,1.548498
267215,0.00,5.416667
259342,0.00,3.035714
2228106,10.00,3.932749
...,...,...
3320590,0.00,5.325077
3939480,0.00,6.328125
6009549,3.00,9.603175
5214264,3.28,6.510417


<class 'pandas.core.frame.DataFrame'>
Index: 6658427 entries, 3818891 to 395563
Data columns (total 2 columns):
 #   Column        Dtype  
---  ------        -----  
 0   tips          float64
 1   min_per_mile  float64
dtypes: float64(2)
memory usage: 152.4 MB


### Step 4 — Diagnose Multicollinearity on **TRAIN** — Correlation Matrix
- Flag any |r| > 0.70 as a potential problem.


In [292]:
# correlation matrix of the train data to be able to see correlations between our models X
corr_matrix = X_train.corr()

print("Correlation Matrix: \n",corr_matrix)
print('\nOur correlation is very low between our features. (-0.031801)')

Correlation Matrix: 
                   tips  min_per_mile
tips          1.000000     -0.031801
min_per_mile -0.031801      1.000000

Our correlation is very low between our features. (-0.031801)


### Step 5 — Diagnose Multicollinearity on **TRAIN** — VIF
- 1–5 normal; >5–10+ concerning.

In [294]:
# using VIF to be confirm correlation matrix, another way to verify if our X's have any correlation
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_train_vif = X_train.copy()

X_train_vif = sm.add_constant(X_train_vif)

In [295]:
# creating DataFrame for vif's data
vif_data = pd.DataFrame()

#taking al features from X_train.vif data
vif_data['feature'] = X_train_vif.columns

# list comprehension to take all values from X_train_vif and perform VIF
vif_data['VIF'] = [variance_inflation_factor(X_train_vif.values, i) for i in range(len(X_train_vif.columns))]

print("\n--- VIF Scores ---")
vif_data


--- VIF Scores ---


,feature,VIF
0,const,1.518206
1,tips,1.001012
2,min_per_mile,1.001012


### Step 6 — Fix High VIF (if needed)

- If two predictors are highly correlated, **drop/choose** using business logic (e.g., keep the more actionable one).
- Recompute VIF to confirm improvement.

My VIFs are a lot lower than 5, showing the independence of both variables. This means it is okay to continue using them in our model as long as they are normal.

### Step 7 —  Fit on TRAIN Only, Predict on TEST, Evaluate MAE/RMSE

- Add intercept (`sm.add_constant`).
- Report **MAE/RMSE** in **units of Y**.
- Also capture **Adjusted R²** from the TRAIN fit summary to comment on fit (don’t use it alone for selection).


In [296]:
np.random.seed(17)

xconstant = sm.add_constant(x)
Xmodel_train, Xmodel_test, ymodel_train, ymodel_test = train_test_split(xconstant, y, test_size=0.2, random_state=17)

print(f"X Training data shape: {Xmodel_train.shape}")
print(f"X Testing data shape:  {Xmodel_test.shape}")
print(f"Y Training data shape: {ymodel_train.shape}")
print(f"Y Testing data shape:  {ymodel_test.shape}")

model = sm.OLS(ymodel_train, Xmodel_train).fit()

X Training data shape: (6658427, 3)
X Testing data shape:  (1664607, 3)
Y Training data shape: (6658427,)
Y Testing data shape:  (1664607,)


In [298]:
predict = model.predict(Xmodel_test)

mae = mean_absolute_error(ymodel_test, predict)
rmse = np.sqrt(mean_squared_error(ymodel_test, predict))

print(f"\nMean Absolute Error (MAE) on Test Data: {mae:.2f}")
print(f"Root Mean Squared Error (RMSE) on Test Data: {rmse:.2f}")
print(f"\nInterpretation: Our model's predictions on new data are off by an average of ${mae:,.2f} shown by our MAE or ${rmse:.2f} shown by our RMSE.")

df['base_passenger_fare'].quantile(.25)

print(f"\nWith the 25th percentile of base_passenger_fare being $11.64, our RMSE and MAE have too great of a margin of error to continue with this model.")


Mean Absolute Error (MAE) on Test Data: 11.83
Root Mean Squared Error (RMSE) on Test Data: 18.73

Interpretation: Our model's predictions on new data are off by an average of $11.83 shown by our MAE or $18.73 shown by our RMSE.

With the 25th percentile of base_passenger_fare being $11.64, our RMSE and MAE have too great of a margin of error to continue with this model.


In [299]:
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                             OLS Regression Results                            
===============================================================================
Dep. Variable:     base_passenger_fare   R-squared:                       0.159
Model:                             OLS   Adj. R-squared:                  0.159
Method:                  Least Squares   F-statistic:                 6.303e+05
Date:                 Thu, 06 Nov 2025   Prob (F-statistic):               0.00
Time:                         19:18:12   Log-Likelihood:            -2.8948e+07
No. Observations:              6658427   AIC:                         5.790e+07
Df Residuals:                  6658424   BIC:                         5.790e+07
Df Model:                            2                                         
Covariance Type:             nonrobust                                         
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           22.1475      0.009   2492.235      0.000      22.130      22.165
tips             2.4906      0.002   1081.862      0.000       2.486       2.495
min_per_mile    -0.2177      0.001   -266.370      0.000      -0.219      -0.216
==============================================================================
Omnibus:                  8074968.513   Durbin-Watson:                   1.999
Prob(Omnibus):                  0.000   Jarque-Bera (JB):      12939715021.075
Skew:                           5.628   Prob(JB):                         0.00
Kurtosis:                     218.671   Cond. No.                         12.8
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In combination with what we've noticed in the relatively high MAE and RMSE, we can see the adjusted R² being low  (0.159). Our y-intercept accounts for when our minutes per mile and tips are at 0, and it is relatively large (close to the 25th quartile). This could potentially mean there are other features within our data that can explain our customers' base pay better than the two I've used in this model.

### Step 8 —  Interpret Coefficients (Plain Language)
Write **unit-based** sentences “**holding others constant**.” Example templates (edit with your β values/units):

- **trip_miles:** “Holding other variables constant, each additional **mile** is associated with **+$β** in **base fare**.”
- **trip_time_minutes:** “Holding others constant, each additional **minute** is associated with **+$β** in **base fare**.”
- **tolls / tips:** interpret as “per $1 change,” holding others constant.

Also note **p-values** and whether they support including each predictor.

- Holding minutes per mile constant, each additional tip is associated with an increase of base passenger fare by $2.49.
- Holding tip constant, each additional mile is associated with a decrease of $-0.217. Take into account our R squared of 0.159, which would say it isn't the best representation of base passenger fare.

- Our p-value is lower than 0.000, showing that these results are not up to chance.

## We Share — Reflection & Wrap‑Up

Write **2 short paragraphs** and be specific:

1) **What changes did you make to handle multicollinearity and why?**  
Reference **corr**/**VIF** on TRAIN and any features you dropped or kept (with business rationale). Include **Adjusted R² (TRAIN)** and **TEST MAE/RMSE**.

A change I had made originially was when I used trip miles and trip_time_minutes as my X. After I saw a correlation above .80, I spoke with Luc and they said potentially joining both variables. After joining both variables and incorporating tip as an X2, I was able to find two variables that avoid multicollinearity. Although the Adjusted R^2 was very low, our y-intercept told us a lot about models in the future and the potential of using more variables with our MLR.

2) **Stakeholder summary (units, one sentence):**  
Give a plain-English takeaway: e.g., “On unseen July trips, our typical error is about **$X** per fare; each extra mile adds about **$β_mile**, holding other factors constant.”

At the start of a trip, a driver can typically expect $22.14. As our driver continues to increase their mileage a minute, expect to lose 21 cents, while holding our tips constant.